In [1]:
# Web Scraper for National Assembly Hansards
# Author: Joshua Been
# Date: August 5, 2025

# This script automates the process of downloading Hansard PDF documents
# from the official Parliament of Kenya website for the National Assembly.
# It is designed to be robust, automatically discovering all available parliamentary
# sessions and pages to ensure a complete collection of documents.

In [2]:
# --- Core Libraries ---
import os                   # For interacting with the operating system (e.g., creating directories, checking for files).
import requests             # For making HTTP requests to fetch web pages.
from bs4 import BeautifulSoup # For parsing HTML content and extracting data.
from urllib.parse import unquote, urlparse, parse_qs, urljoin # For handling and parsing URLs.
import time                 # For adding delays to be respectful to the web server.
from pathlib import Path    # For setting download directories

In [3]:
# ==============================================================================
# --- CONFIGURATION SETTINGS ---
# ==============================================================================

# The base URL of the website, used for constructing absolute URLs from relative ones.
BASE_URL = "https://www.parliament.go.ke"
# The starting page that lists the Hansard documents.
START_URL = "https://www.parliament.go.ke/the-national-assembly/house-business/hansard"
# The name of the local directory where the downloaded PDFs will be saved.
# Get the current notebook's directory
notebook_dir = Path.cwd()
# Go one level up, then into 'national_assembly_hansards'
DOWNLOAD_DIR = notebook_dir.parent / "inputs" /"national_assembly_hansards"


In [4]:
# ==============================================================================
# --- SCRIPT EXECUTION ---
# ==============================================================================

# --- Step 1: Setup the Download Directory ---
# Check if the target directory exists. If not, create it.
# This prevents errors when the script tries to save files.
if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)

In [5]:
# --- Step 2: Dynamically Discover Available Parliamentary Sessions ---
# Instead of hardcoding session years, the script first visits the main Hansard
# page to find all available sessions listed in the filter dropdown menu.
# This makes the script future-proof, as it will automatically find new sessions
# as they are added to the website.
session_values = []
try:
    print("Finding available parliamentary sessions...")
    main_page_response = requests.get(START_URL)
    # Raise an exception if the request was unsuccessful (e.g., 404 Not Found).
    main_page_response.raise_for_status()
    soup = BeautifulSoup(main_page_response.text, 'html.parser')

    # Find the HTML <select> element (dropdown menu) by its 'name' attribute.
    session_dropdown = soup.find('select', {'name': 'field_parliament_value'})
    if session_dropdown:
        # Extract the 'value' from each <option> tag within the dropdown.
        # The first option is usually a placeholder like "- Any -", so it's skipped.
        for option in session_dropdown.find_all('option')[1:]:
            if option.has_attr('value'):
                session_values.append(option['value'])
    print(f"Found sessions: {session_values}")
except requests.exceptions.RequestException as e:
    print(f"Could not fetch parliamentary sessions: {e}")

Finding available parliamentary sessions...
Found sessions: ['2013', '2017', '2018', '2019', '2020', '2021', '2022']


In [6]:
# --- Step 3: Iterate Through Each Session and Download PDFs ---
# This block iterates through each parliamentary session discovered in the initial setup.

for session in session_values:
    print(f"\n--- Starting session: {session} ---")

    # --- Step 3a: Dynamically Determine the Number of Pages for the Session ---
    # This section implements a robust, multi-stage process to accurately determine
    # the total number of pages for a given session, accounting for variations in
    # the website's HTML structure.

    last_page = 0
    try:
        # Construct and fetch the URL for the first page of the current session.
        first_page_url = f"{START_URL}?title=%20&field_parliament_value={session}&page=0"
        response = requests.get(first_page_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # --- Hierarchical Pagination Logic ---
        # This logic attempts to find the last page number using a prioritized
        # series of methods to ensure maximum reliability across different website formats.
        found_last_page = False

        # Method 1: Primary strategy is to locate the "Last" button link using the
        # class name 'pager__item--last', which is the most common format observed.
        last_page_li = soup.find('li', class_='pager__item--last')
        if last_page_li and last_page_li.find('a'):
            href = last_page_li.find('a')['href']
            query_params = parse_qs(urlparse(href).query)
            if 'page' in query_params:
                last_page = int(query_params['page'][0])
                found_last_page = True

        # Method 2: If the primary method fails, this secondary strategy attempts
        # to find the "Last" button using an alternative class name, 'pager-last'.
        if not found_last_page:
            last_page_li = soup.find('li', class_='pager-last')
            if last_page_li and last_page_li.find('a'):
                href = last_page_li.find('a')['href']
                query_params = parse_qs(urlparse(href).query)
                if 'page' in query_params:
                    last_page = int(query_params['page'][0])
                    found_last_page = True

        # Method 3: As a final fallback, if no "Last" button is found, this method
        # collects all visible page numbers and uses the maximum value. This is
        # effective for sessions with a small number of pages where a "Last" link is not rendered.
        if not found_last_page:
            # Selects all list items from two potential parent class names for the pager.
            list_items = soup.select('ul.pager li, ul.pager__items li')
            pages = [0] # Initialize with 0 to handle single-page sessions.
            for item in list_items:
                text = item.get_text(strip=True)
                if text.isdigit():
                    # The visible page numbers are 1-indexed, while the URL parameter is 0-indexed.
                    # Convert the visible number to its 0-indexed equivalent for the loop.
                    pages.append(int(text) - 1)
            if pages:
                last_page = max(pages)

        print(f"Found {last_page + 1} pages for this session.")

    except requests.exceptions.RequestException as e:
        print(f"Could not determine page count for session {session}: {e}")
        continue

    # --- Step 3b: Loop Through Each Page and Scrape PDF Links ---

    # Iterates from the first page (0) to the determined last page.
    for page_num in range(last_page + 1):
        page_url = f"{START_URL}?title=%20&field_parliament_value={session}&page={page_num}"
        print(f"Scraping page: {page_url}")

        try:
            response = requests.get(page_url)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            # Precisely targets the table cells containing PDF links.
            for td in soup.find_all('td', class_='views-field-field-pdf'):
                link = td.find('a')
                if link and 'href' in link.attrs:
                    pdf_url = link['href']
                    # Ensure the URL is absolute before making a request.
                    pdf_url = urljoin(BASE_URL, pdf_url)
                    # Extract a clean filename from the URL.
                    filename = os.path.basename(unquote(pdf_url))
                    filepath = os.path.join(DOWNLOAD_DIR, filename)

                    # To optimize performance and bandwidth, skip files that already exist.
                    if not os.path.exists(filepath):
                        print(f"  Downloading: {filename}")
                        pdf_response = requests.get(pdf_url)
                        pdf_response.raise_for_status()
                        # Save the file in binary write mode.
                        with open(filepath, 'wb') as f:
                            f.write(pdf_response.content)
                        # A politeness delay to prevent overwhelming the server.
                        time.sleep(1)

                    else:
                        print(f"  Skipping (already exists): {filename}")

        except requests.exceptions.RequestException as e:
            # Log errors for individual pages but continue the process for other pages.
            print(f"An error occurred while scraping page {page_num}: {e}")

print("\nScraping process completed.")


--- Starting session: 2013 ---
Found 17 pages for this session.
Scraping page: https://www.parliament.go.ke/the-national-assembly/house-business/hansard?title=%20&field_parliament_value=2013&page=0
  Skipping (already exists): ORDER PAPER FOR TUESDAY, 4TH NOVEMBER 2025.pdf
  Skipping (already exists): Hansard Report - Thursday, 16th October 2025 (P).pdf
  Skipping (already exists): Hansard Report - Thursday, 16th October 2025 (A).pdf
  Skipping (already exists): Hansard Report - Wednesday, 15th October 2025 (P).pdf
  Skipping (already exists): Hansard_Report_-_Tuesday_18th_June_2013_-_Afternoon.pdf
  Skipping (already exists): Hansard_report_-_Thursday_13th_June_2013_-_Afternoon.pdf
  Skipping (already exists): Hansard_Report_-_Wednesday_12th_June_2013_Afternoon_1.pdf
  Skipping (already exists): Hansard_Report_-_Wednesday_12th_June_2013_-_Morning.pdf
  Skipping (already exists): Hansard_Report_-_Tuesday_11th_June_2013_Afternoon.pdf
  Skipping (already exists): Hansard_Report_-_Wednes

KeyboardInterrupt: 